# Data Acquisition

In [ ]:
import pandas as pd
import numpy as np
import os
import urllib.request
from google.colab import files

# --- 1. SETUP ---
print("🏗️  Setting up Data Folders...")
BASE_DIR = os.getcwd()
RAW_DIR = os.path.join(BASE_DIR, 'data', '01_raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', '02_processed')

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# --- 2. DOWNLOAD RAW DATA ---
print("\n⬇️  Step 1: Downloading Raw Data from UCI...")
URLS = {
    'processed.cleveland.data': 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data',
    'processed.hungarian.data': 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.hungarian.data',
    'processed.switzerland.data': 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.switzerland.data',
    'processed.va.data': 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.va.data'
}

for filename, url in URLS.items():
    destination = os.path.join(RAW_DIR, filename)
    try:
        urllib.request.urlretrieve(url, destination)
        print(f"   ✅ Downloaded: {filename}")
    except Exception as e:
        print(f"   ❌ Failed: {filename}")

# --- 3. MERGE & CLEAN ---
print("\n⚙️  Step 2: Merging & Cleaning Data...")
COL_NAMES = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", 
             "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"]

all_dfs = []
for filename in URLS.keys():
    file_path = os.path.join(RAW_DIR, filename)
    if os.path.exists(file_path):
        df = pd.read_csv(file_path, names=COL_NAMES, na_values=["?", "-9.0", "-9"])
        all_dfs.append(df)

df_combined = pd.concat(all_dfs, ignore_index=True)

# Impute Missing Values
for col in df_combined.columns:
    if df_combined[col].dtype in ['float64', 'int64']:
        df_combined[col] = df_combined[col].fillna(df_combined[col].median())
    else:
        df_combined[col] = df_combined[col].fillna(df_combined[col].mode()[0])

# Binarize Target (0=Healthy, 1-4=Disease)
df_combined['target'] = df_combined['target'].apply(lambda x: 1 if x > 0 else 0)

# --- 4. SAVE & DOWNLOAD ---
processed_path = os.path.join(PROCESSED_DIR, 'heart_disease_combined.csv')
df_combined.to_csv(processed_path, index=False)

print(f"✅ Processed data saved ({len(df_combined)} rows) to: {processed_path}")
print("⬇️  Triggering download...")
files.download(processed_path)

⬇️  Step 1: Downloading Raw Data from UCI...
   ✅ Downloaded: processed.cleveland.data
   ✅ Downloaded: processed.hungarian.data
   ✅ Downloaded: processed.switzerland.data
   ✅ Downloaded: processed.va.data

⚙️  Step 2: Merging & Cleaning Data...
------------------------------
✅ SUCCESS! Created dataset with 920 rows.
⬇️  Triggering download to your laptop...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>